To begin, we load the dataset and perform basic exploration to understand the shape and timeframe of our data. We will check the total number of matches, the date range, the number of unique countries that have played, and the most frequent home team.

In [ ]:
import pandas as pd

# Load the CSV
df = pd.read_csv("results.csv")

# 1. How many matches are in the dataset?
total_matches = df.shape[0]
print(f"Total matches: {total_matches}")

# 2. What is the earliest and latest year in the data?
# Convert date column to datetime objects
df["date"] = pd.to_datetime(df["date"])
earliest_year = df["date"].dt.year.min()
latest_year = df["date"].dt.year.max()
print(f"Earliest year: {earliest_year}")
print(f"Latest year: {latest_year}")

# 3. How many unique countries are there?
# Combine home and away teams to get all unique participating countries
unique_countries = pd.concat([df["home_team"], df["away_team"]]).nunique()
print(f"Unique countries: {unique_countries}")

# 4. Which team appears most frequently as home team?
most_freq_home = df["home_team"].value_counts().head(1).index[0]
print(f"Most frequent home team: {most_freq_home}")

Next, we analyze the scoring trends. By creating a total_goals column, we can calculate the average goals per match, identify the highest-scoring match in history, compare home versus away scoring, and find the most common number of goals scored in a single game.

In [ ]:
# Create total goals
df["total_goals"] = df["home_score"] + df["away_score"]

# 1. What is the average number of goals per match?
avg_goals = df["total_goals"].mean()
print(f"Average goals per match: {avg_goals:.2f}")

# 2. What is the highest scoring match?
highest_scoring_index = df["total_goals"].idxmax()
highest_match = df.loc[highest_scoring_index]
print(f"Highest scoring match: {highest_match['home_team']} {highest_match['home_score']} - {highest_match['away_score']} {highest_match['away_team']} on {highest_match['date'].date()}")

# 3. Are more goals scored at home or away?
total_home_goals = df["home_score"].sum()
total_away_goals = df["away_score"].sum()
print(f"Total Home Goals: {total_home_goals}")
print(f"Total Away Goals: {total_away_goals}")
if total_home_goals > total_away_goals:
    print("More goals are scored at HOME.")
else:
    print("More goals are scored AWAY.")

# 4. What is the most common total goals value?
most_common_goals = df["total_goals"].mode()[0]
print(f"Most common total goals value: {most_common_goals}")

To analyze match outcomes, we define a function to categorize each match as a "Home Win", "Away Win", or "Draw". We apply this to evaluate the existence of a home advantage and to determine which country has secured the most historical victories.

In [ ]:
# Create match outcome
def match_result(row):
    if row["home_score"] > row["away_score"]:
        return "Home Win"
    elif row["home_score"] < row["away_score"]:
        return "Away Win"
    else:
        return "Draw"

df["result"] = df.apply(match_result, axis=1)

# 1. What percentage of matches are home wins?
home_win_pct = (df["result"] == "Home Win").mean() * 100
print(f"Percentage of home wins: {home_win_pct:.2f}%")

# 2. Does home advantage exist?
away_win_pct = (df["result"] == "Away Win").mean() * 100
draw_pct = (df["result"] == "Draw").mean() * 100
print(f"Home Win %: {home_win_pct:.2f}%, Away Win %: {away_win_pct:.2f}%, Draw %: {draw_pct:.2f}%")
print("Yes, home advantage exists, as the home win percentage is significantly higher than the away win percentage.")

# 3. Which country has the most wins historically?
home_wins = df[df["result"] == "Home Win"]["home_team"].value_counts()
away_wins = df[df["result"] == "Away Win"]["away_team"].value_counts()
total_wins = home_wins.add(away_wins, fill_value=0)
most_wins_country = total_wins.idxmax()
print(f"Country with the most wins historically: {most_wins_country} ({int(total_wins.max())} wins)")

Finally, we visualize our findings using Matplotlib to create a histogram of goal distributions, a bar chart of match outcomes, and a bar chart showing the top 10 most successful teams.

In [ ]:
import matplotlib.pyplot as plt

# 1. Histogram of goals
plt.figure(figsize=(8, 5))
df["total_goals"].hist(bins=15, color='skyblue', edgecolor='black')
plt.title("Distribution of Goals Per Match")
plt.xlabel("Total Goals")
plt.ylabel("Frequency")
plt.show()

# 2. Bar chart of match outcomes
plt.figure(figsize=(8, 5))
df["result"].value_counts().plot(kind='bar', color=['green', 'red', 'gray'])
plt.title("Match Outcomes Distribution")
plt.xlabel("Outcome")
plt.ylabel("Number of Matches")
plt.xticks(rotation=0)
plt.show()

# 3. Top 10 teams by total wins
plt.figure(figsize=(10, 6))
top_10_teams = total_wins.sort_values(ascending=False).head(10)
top_10_teams.plot(kind='bar', color='orange', edgecolor='black')
plt.title("Top 10 Teams by Total Wins")
plt.xlabel("Country")
plt.ylabel("Total Wins")
plt.xticks(rotation=45)
plt.show()